# Import necessary libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE
from IPython.display import display

# Read file with Labels

In [2]:
dataset = pd.read_csv('conversion_data_train.csv')
print('Set with labels (our train+test) :', dataset.shape)

Set with labels (our train+test) : (284580, 6)


# Explore DATASET

In [3]:
dataset.head(10)

,country,age,new_user,source,total_pages_visited,converted
0,China,22,1,Direct,2,0
1,UK,21,1,Ads,3,0
2,Germany,20,0,Seo,14,1
3,US,23,1,Seo,3,0
4,US,28,1,Direct,3,0
5,US,29,0,Seo,7,0
6,US,30,1,Direct,4,0
7,UK,38,1,Ads,2,0
8,UK,26,1,Seo,4,0
9,UK,31,0,Seo,5,0


In [4]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 284580 entries, 0 to 284579
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   country              284580 non-null  str  
 1   age                  284580 non-null  int64
 2   new_user             284580 non-null  int64
 3   source               284580 non-null  str  
 4   total_pages_visited  284580 non-null  int64
 5   converted            284580 non-null  int64
dtypes: int64(4), str(2)
memory usage: 13.0 MB


In [5]:
dataset.describe(include='all')


,country,age,new_user,source,total_pages_visited,converted
count,284580,284580.000000,284580.000000,284580,284580.000000,284580.000000
unique,4,NaN,NaN,3,NaN,NaN
top,US,NaN,NaN,Seo,NaN,NaN
freq,160124,NaN,NaN,139477,NaN,NaN
mean,NaN,30.564203,0.685452,NaN,4.873252,0.032258
std,NaN,8.266789,0.464336,NaN,3.341995,0.176685
min,NaN,17.000000,0.000000,NaN,1.000000,0.000000
25%,NaN,24.000000,0.000000,NaN,2.000000,0.000000
50%,NaN,30.000000,1.000000,NaN,4.000000,0.000000
75%,NaN,36.000000,1.000000,NaN,7.000000,0.000000


In [6]:
dataset.shape

(284580, 6)

In [7]:
dataset.converted.value_counts()

converted
0    275400
1      9180
Name: count, dtype: int64

In [8]:
#le dataset est extrêmement déséquilibré: 97% de non conversion contre 3% de conversion
dataset['converted'].value_counts(normalize=True)



converted
0    0.967742
1    0.032258
Name: proportion, dtype: float64

# BASELINE

In [9]:
#séparer les features de la target
X = dataset.drop('converted', axis=1)
y = dataset['converted']

In [10]:
#stratify to maintain the same proportion of classes in train and val sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [11]:
#preprocessing

numerical_features = ['age', 'total_pages_visited']
categorical_features = ['country', 'source', 'new_user']


preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


In [12]:
#Baseline Logistic Regression model as a reference
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [13]:
#Evaluation on the validation set


y_val_pred = model.predict(X_val)
y_val_proba = model.predict_proba(X_val)[:, 1]

print("Accuracy :", accuracy_score(y_val, y_val_pred))
print("ROC-AUC  :", roc_auc_score(y_val, y_val_proba))
print("Log loss :", log_loss(y_val, y_val_proba))


Accuracy : 0.9865767095368614
ROC-AUC  : 0.9868995958344606
Log loss : 0.039711159747872385


Ce modèle :

Accuracy ≈ 98.7%

AUC ≈ 0.987

Log loss très faible

L’accuracy est élevée mais peu informative seule à cause du déséquilibre.
Interprétation forte :

Le modèle est excellent pour distinguer les utilisateurs qui vont convertir de ceux qui ne convertiront pas.

Concrètement :

Si je prends un converti et un non-converti au hasard,

Le modèle a 98.7% de chances de donner un score plus élevé au converti.

👉 C’est un score de très haute qualité.

Log loss ≈ 0.039 (très faible)==> Le modèle produit des probabilités très bien calibrées.Ça veut dire :

Il n’est pas seulement “bon en classement”
Il est prudent quand il doute
Il est confiant quand il a raison



La baseline logistic regression donne déjà des performances très élevées. Le ROC-AUC proche de 0.99 montre une excellente capacité de discrimination, et le log loss très faible indique que les probabilités sont bien calibrées. Cela constitue une base solide pour comparer des modèles plus complexes.”

## Random Forest

In [14]:
from sklearn.ensemble import RandomForestClassifier


In [15]:
#Pipeline RandomForest
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])


In [16]:
#Entrainement du modèle RandomForest
rf_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [17]:
#Evaluation 
y_val_pred_rf = rf_model.predict(X_val)
y_val_proba_rf = rf_model.predict_proba(X_val)[:, 1]

print("Accuracy RF :", accuracy_score(y_val, y_val_pred_rf))
print("ROC-AUC RF  :", roc_auc_score(y_val, y_val_proba_rf))
print("Log loss RF :", log_loss(y_val, y_val_proba_rf))


Accuracy RF : 0.9849602923606718
ROC-AUC RF  : 0.9516322218187687
Log loss RF : 0.1394695569782093


Les Scores obtenus sont similaires à la logistic. Le RandomForest n'apporte pas de'amélioration sans feature Engineering.
Log loss de 0,03 à 0,13 est mauvaise. Ça signifie :RandomForest est mal calibré.
Le RandomForest n’est PAS meilleur que la Logistic Regression dans cet état.
Il n’apporte aucun gain de discrimination (AUC identique) et dégrade fortement la qualité des probabilités (log loss beaucoup plus élevé).
Donc il est objectivement inférieur.


# Feature Engineering

In [18]:
#creation of age bins as a new feature
X = dataset.drop('converted', axis=1)
y = dataset['converted']

X_fe = X.copy()

X_fe['age_bin'] = pd.cut(
    X_fe['age'],
    bins=[0, 25, 35, 45, 60, 100],
    labels=['<25', '25-35', '35-45', '45-60', '60+']
)


In [19]:
#creation of total_pages_visited bins as a new feature
X_fe['pages_bin'] = pd.cut(
    X_fe['total_pages_visited'],
    bins=[0, 2, 5, 10, 20, 100],
    labels=['1-2', '3-5', '6-10', '11-20', '20+']
)


In [20]:
#creation of grouped country feature
country_counts = X_fe['country'].value_counts(normalize=True)
rare_countries = country_counts[country_counts < 0.05].index

X_fe['country_grouped'] = X_fe['country'].apply(
    lambda x: 'Other' if x in rare_countries else x
)


In [21]:
X_fe['country_grouped'].value_counts()

country_grouped
US       160124
China     69122
UK        43641
Other     11693
Name: count, dtype: int64

In [22]:
#creation of interaction feature between source and new_user
X_fe['source_new_user'] = (
    X_fe['source'] + '_' + X_fe['new_user'].astype(str)
)


In [23]:
#new numerical features list
num_features_fe = ['age', 'total_pages_visited']



In [24]:
#new categorical features list
cat_features_fe = [
    'country_grouped',
    'source',
    'new_user',
    'age_bin',
    'pages_bin',
    'source_new_user'
]


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_fe, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [26]:
#new preprocessor with the new features
preprocessor_fe = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features_fe),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features_fe)
    ]
)


In [27]:
#logistic regression model with new features
model_fe = Pipeline(steps=[
    ('preprocessor', preprocessor_fe),
    ('classifier', LogisticRegression(max_iter=1000))
])

model_fe.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [28]:
#Evaluation
y_val_pred_fe = model_fe.predict(X_val)
y_val_proba_fe = model_fe.predict_proba(X_val)[:, 1]

print("Accuracy FE :", accuracy_score(y_val, y_val_pred_fe))
print("ROC-AUC FE  :", roc_auc_score(y_val, y_val_proba_fe))
print("Log loss FE :", log_loss(y_val, y_val_proba_fe))


Accuracy FE : 0.986348302761965
ROC-AUC FE  : 0.9868820139610754
Log loss FE : 0.03975237072196751


Tu as (avec feature engineering) :
Accuracy FE = 0.98635
ROC-AUC FE = 0.98688
Log loss FE = 0.03975
Et ta baseline logistic (avant FE) était environ :
Accuracy ≈ 0.98658
ROC-AUC ≈ 0.98690
Log loss ≈ 0.03971

==> Résultat : c’est quasiment identique, et même légèrement pire (mais la différence est minuscule).

# Cross-validation

In [29]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'neg_log_loss': 'neg_log_loss'
}

res = cross_validate(model_fe, X_fe, y, cv=cv, scoring=scoring, n_jobs=-1)

print("Accuracy CV:", np.mean(res['test_accuracy']), "+/-", np.std(res['test_accuracy']))
print("AUC CV     :", np.mean(res['test_roc_auc']), "+/-", np.std(res['test_roc_auc']))
print("LogLoss CV :", -np.mean(res['test_neg_log_loss']), "+/-", np.std(-res['test_neg_log_loss']))


Accuracy CV: 0.9861515215405159 +/- 0.0005479284251746152
AUC CV     : 0.9858511653874815 +/- 0.0009916399784113102
LogLoss CV : 0.04064173271412669 +/- 0.000804807450484637


Interprétation de ta cross-val (ce que ça dit vraiment)

On a en moyenne (5-fold CV) :Accuracy ≈ 0.98615 ± 0.00055 \\ AUC ≈ 0.98585 ± 0.00099 \\ LogLoss ≈ 0.04064 ± 0.00080
Le modèle est très stable (écarts-types très faibles).Les scores sur un seul split étaient un peu optimistes (normal).
Le log loss ~0.0406 est cohérent avec un très bon modèle dans ce dataset.
Et surtout :
👉 À ce niveau, un gain “réel” doit dépasser le bruit, donc typiquement :
AUC +0.002 ou +0.003 (et répété en CV) = vrai progrès
LogLoss -0.001 ou -0.002 (et répété en CV) = vrai progrès

# HistGradientBoostingClassifier HGB + CV

In [30]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

num_features_fe = ['age', 'total_pages_visited']
cat_features_fe = ['country', 'source', 'new_user']  # commence simple (sans bins) pour tester

preprocessor_hgb = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features_fe),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features_fe)
    ]
)


In [31]:
#MODEL HGB
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=300,
    random_state=42
)

model_hgb = Pipeline(steps=[
    ('preprocessor', preprocessor_hgb),
    ('classifier', hgb)
])



In [32]:
# CV
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'neg_log_loss': 'neg_log_loss'
}

res = cross_validate(model_hgb, X, y, cv=cv, scoring=scoring, n_jobs=-1)

print("Accuracy CV:", np.mean(res['test_accuracy']), "+/-", np.std(res['test_accuracy']))
print("AUC CV     :", np.mean(res['test_roc_auc']), "+/-", np.std(res['test_roc_auc']))
print("LogLoss CV :", -np.mean(res['test_neg_log_loss']), "+/-", np.std(-res['test_neg_log_loss']))


Accuracy CV: 0.9860812425328553 +/- 0.0006595002449548498
AUC CV     : 0.9856208072472917 +/- 0.00107577307973733
LogLoss CV : 0.04098850678957012 +/- 0.0007864141964863


Après comparaison par cross-validation, les modèles plus complexes (RandomForest, GradientBoosting) n’apportent pas de gain significatif et dégradent légèrement le log loss. La régression logistique offre le meilleur compromis performance / stabilité / interprétabilité.

##############################
# FINAL MODEL FOR SUBMISSION
##############################


In [45]:
;#definition des features et target
X = dataset.drop('converted', axis=1)
y = dataset['converted']


In [48]:
#Final preprocessor
num_features = ['age', 'total_pages_visited']
cat_features = ['country', 'source', 'new_user']

preprocessor_final = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)


In [49]:
#final model
final_model = Pipeline(steps=[
    ('preprocessor', preprocessor_final),
    ('classifier', LogisticRegression(max_iter=1000))
])


In [50]:
#training final model on the entire dataset (data_train.csv)
final_model.fit(X, y)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [51]:
#predicting conversion probabilities on the test data
test_proba = final_model.predict_proba(test_data)[:, 1]


In [52]:
#transforming probabilities to binary predictions with a threshold of 0.5
test_pred = (test_proba >= 0.5).astype(int)


In [54]:
#creating the submission file
submission = pd.DataFrame({
    "converted": test_pred
})

submission.to_csv(
    "conversion_data_test_predictions_Semia-Model1.csv",
    index=False
)


In [55]:
#verifying the submission file
submission.head(10)
submission['converted'].value_counts()


converted
0    30798
1      822
Name: count, dtype: int64

In [37]:
#reading the test data
test_data = pd.read_csv("conversion_data_test.csv")


In [38]:
#predicting conversion probabilities on the test data
test_proba = final_model.predict_proba(test_data)[:, 1]


In [39]:
#creating the submission file
submission = pd.DataFrame({
    "converted": test_proba
})

submission.to_csv(
    "conversion_data_test_predictions_Semia-model.csv",
    index=False
)


In [40]:
submission.head()
submission.shape


(31620, 1)

##############################
# PART 2 - F1 + Confusion Matrix c'est juste pour s'assurer qu'on a bien fait les choses
##############################


In [41]:
X = dataset.drop('converted', axis=1)
y = dataset['converted']


In [42]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [43]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

num_features = ['age', 'total_pages_visited']
cat_features = ['country', 'source', 'new_user']

preprocessor_final = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

final_model = Pipeline(steps=[
    ('preprocessor', preprocessor_final),
    ('classifier', LogisticRegression(max_iter=1000))
])

final_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [44]:
from sklearn.metrics import f1_score, confusion_matrix, classification_report

y_val_proba = final_model.predict_proba(X_val)[:, 1]
y_val_pred = (y_val_proba >= 0.5).astype(int)

print("F1-score (threshold=0.5):", f1_score(y_val, y_val_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, y_val_pred))
print("\nClassification report:\n", classification_report(y_val, y_val_pred))


F1-score (threshold=0.5): 0.7684848484848484
Confusion matrix:
 [[54884   196]
 [  568  1268]]

Classification report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99     55080
           1       0.87      0.69      0.77      1836

    accuracy                           0.99     56916
   macro avg       0.93      0.84      0.88     56916
weighted avg       0.99      0.99      0.99     56916

